# Master analysis notebook

Canonical, readable reconstruction of the scientific workflow for the alginate-hyaluronic acid bioink reanalysis. This notebook uses only repository-relative paths, executes the canonical analysis scripts in dependency order, and reads the outputs generated in the current run. Heavy implementations remain in `src/`; the notebook exposes the inputs, assumptions, validation design, and reportable outputs rather than duplicating the full source code.

**Execution rule:** run from a fresh kernel using **Run All**. No cell depends on manual state from an earlier session. The retrospective candidate-ranking bootstrap is the only documented analysis that reads a historical artifact under `archive/`.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys, time
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Resolve repository root robustly whether Jupyter starts in root or notebooks/.
HERE = Path.cwd().resolve()
ROOT = HERE if (HERE / 'src').is_dir() else HERE.parent
assert (ROOT / 'src').is_dir(), f'Repository root not found from {HERE}'
SEED = 42
np.random.seed(SEED)

def run_script(relpath):
    path = ROOT / relpath
    assert path.exists(), f'Missing canonical script: {relpath}'
    t0 = time.perf_counter()
    p = subprocess.run([sys.executable, str(path)], cwd=ROOT, text=True, capture_output=True)
    dt = time.perf_counter()-t0
    if p.returncode != 0:
        print(p.stdout)
        print(p.stderr, file=sys.stderr)
        raise RuntimeError(f'{relpath} failed with exit code {p.returncode}')
    print(f'PASS  {relpath}  ({dt:.1f} s)')
    return p.stdout

def read_csv(relpath, **kwargs):
    path = ROOT / relpath
    assert path.exists(), f'Missing generated output: {relpath}'
    return pd.read_csv(path, **kwargs)

print('Repository root:', ROOT)
print('Python:', sys.version.split()[0])
print('Canonical seed:', SEED)

## 1. Load and curate source data

The authoritative numerical inputs are the three curated source-derived tables corresponding to source Tables S1-S3. This step rebuilds the processed master and target-specific modeling datasets from those tables; it does not use previously generated model outputs.

In [ ]:
run_script('src/prepare_canonical_inputs.py')

source_files = sorted((ROOT/'data').rglob('*.csv'))
print('CSV files under data/:', len(source_files))
for p in source_files:
    if p.name in {'rheology_table_S1.csv','printability_table_S2.csv','angle_fidelity_table_S3.csv'}:
        print(p.relative_to(ROOT))

targets = {}
for target in ['Pr','SR','Qm','AF']:
    p = ROOT/'data'/'processed'/f'modeling_dataset_{target}.csv'
    targets[target] = pd.read_csv(p)
    print(target, targets[target].shape)
    display(targets[target].head(3))

## 2. Feature definitions

Canonical engineered features preserve physically motivated composition, pressure-ratio, and interaction descriptors while excluding exact aliases and deterministic duplicate representations. The model matrices are generated by the preparation module and are inspected here before modeling.

In [ ]:
feature_files = list((ROOT/'data'/'processed').glob('*feature*')) + list((ROOT/'results').glob('**/*feature*definition*'))
print('Feature-definition artifacts found:')
for p in feature_files[:20]: print(' -', p.relative_to(ROOT))

for target, df in targets.items():
    print(f'\n{target}: {len(df.columns)} columns')
    print(', '.join(df.columns))

## 3. Target-specific datasets

Targets are analyzed separately so missing outputs do not force deletion of otherwise valid observations. The expected final sample sizes are 40 (Pr), 43 (SR), 40 (Qm), and 96 (AF).

In [ ]:
expected_n={'Pr':40,'SR':43,'Qm':40,'AF':96}
target_columns={'Pr':'Pr_mean','SR':'SR_mean','Qm':'Qm_mean_mg_s','AF':'AF_mean'}
summary=[]
for t,df in targets.items():
    assert len(df)==expected_n[t], f'{t}: expected {expected_n[t]}, got {len(df)}'
    ycol=target_columns[t]
    assert ycol in df.columns, f'{t}: expected target column {ycol} not found'
    s=df[ycol].describe()[['count','mean','std','min','50%','max']]
    summary.append([t,*s.tolist()])
display(pd.DataFrame(summary, columns=['Target','N','Mean','SD','Min','Median','Max']))


## 4. Core versus engineered representation sensitivity

This analysis is a representation-sensitivity check distinct from the selection-aware nested performance. Six of eight comparisons use the same algorithm on both representations; the two rows with different algorithms are not interpreted as pure feature ablations.

In [ ]:
nested_summary = ROOT/'results'/'revision_comments_4_R1_16_R2'/'Nested_Repeated_Summary_Comments_4-R1_16-R2.csv'
if not nested_summary.exists():
    print('Computational prerequisite: generating canonical nested-validation outputs before the S19 representation analysis.')
    run_script('src/revision_comments_4_R1_16_R2_nested_cv.py')
run_script('src/revision_comments_6_7_R1_8_R2_ablation_selected.py')
s19=read_csv('results/revision_comments_6_7_R1_8_R2/Table_S19_Ablation_Comments_6-7-R1_8-R2.csv')
display(s19)


## 5. Nested repeated cross-validation

Outer five-fold cross-validation is repeated 20 times. Algorithm and eligible feature-set selection occur only within each outer-training partition using inner five-fold validation. Reported fold-wise performance is therefore selection-aware.

In [ ]:
rep=read_csv('results/revision_comments_4_R1_16_R2/Reportable_Nested_Performance_Comments_4-R1_16-R2.csv')
display(rep)


## 6. Nested leave-one-formulation-out validation

For formulation-level transfer, each formulation is reserved as the outer test group. Model and feature-set selection is performed using inner LOFO only on the remaining formulations.

In [ ]:
lofo=read_csv('results/revision_comments_4_R1_16_R2/Nested_LOFO_Summary_Comments_4-R1_16-R2.csv')
display(lofo)
print('Held-out formulation diagnostics:')
lofo_groups=read_csv('results/revision_comments_4_R1_16_R2/Nested_LOFO_Group_Metrics_Comments_4-R1_16-R2.csv')
display(lofo_groups.head(12))

## 7. R² aggregation

R² is non-additive across small test folds. The manuscript therefore reports both the arithmetic mean of the 100 outer-fold R² values and a complementary repeat-pooled calculation that pools all out-of-fold predictions within each of the 20 repetitions.

In [ ]:
run_script('src/revision_comments_4_R1_16_R2_r2_aggregation.py')
r2agg=read_csv('results/revision_comments_4_R1_16_R2/R2_Aggregation_Summary_Comments_4-R1_16-R2.csv')
display(r2agg)

## 8. AF and Qm simple baselines

Reviewer-requested controls test whether the strongest apparent multivariable results require model complexity. AF is evaluated from deposition angle alone; Qm is evaluated from P/TP alone using linear and positive power-law forms.

In [ ]:
run_script('src/revision_comment_3_R2_baselines.py')
baselines=read_csv('results/revision_comment_3_R2/baseline_summary_Comment_3-R2.csv')
display(baselines)

## 9. Baseline versus nested matched comparison

For an apples-to-apples repeated comparison, each simple baseline is fit on the exact outer-test memberships written by the nested procedure, keyed by `(Repeat, Outer_fold, Row_index)`. Baseline and nested predictions are then pooled within the same 20 outer repetitions before R², MAE, and RMSE are calculated.


In [ ]:
run_script('src/revision_comment_3_R2_baseline_contrast.py')
matched=read_csv('results/revision_comment_3_R2_baseline_contrast/Baseline_vs_Nested_Apples_to_Apples_Comment_3-R2.csv')
display(matched)

## 10. Dimensionless Pi_y sensitivity

The reviewer-requested stress-scale control defines Pi_y = applied pressure / formulation-mean yield stress. It is evaluated as a one-descriptor sensitivity and as a one-for-one replacement for P/TP in the matched engineered representation. It is not treated as an exact nozzle-wall shear-stress calculation.

In [ ]:
run_script('src/revision_comment_10_R1_dimensionless_stress_ratio.py')
pi=read_csv('results/revision_comment_10_R1_dimensionless_stress_ratio/Table_S28_Single_Descriptor_Summary.csv')
display(pi)
sub=read_csv('results/revision_comment_10_R1_dimensionless_stress_ratio/Dimensionless_Stress_Ratio_Multivariable_Substitution.csv')
display(sub)

## 11. LOFO uncertainty and formulation-block bootstrap

The evaluation-stage bootstrap resamples the eight already-held-out formulation blocks and recomputes pooled metrics. It quantifies evaluation-sample variability conditional on the fixed nested-LOFO predictions; it does not refit the full nested training procedure.

In [ ]:
run_script('src/revision_comments_12_R1_9_R2_lofo_uncertainty.py')
boot=read_csv('results/revision_comments_12_R1_9_R2/All_Targets_LOFO_Formulation_Block_Bootstrap_Comments_12-R1_9-R2.csv')
display(boot)
srboot=read_csv('results/revision_comments_12_R1_9_R2/SR_LOFO_Formulation_Block_Bootstrap_Comments_12-R1_9-R2.csv')
display(srboot)

## 12. Out-of-fold permutation importance

Permutation importance is calculated only on held-out test observations. Each feature is shuffled within the held-out fold and importance is defined by the increase in test RMSE relative to the unpermuted fold.

In [ ]:
run_script('src/revision_comments_8_R1_17_18_R2_xai.py')
perm=read_csv('results/revision_comments_8_R1_17_18_R2/OOF_Permutation_Repeated_Summary_Comments_18-R2.csv')
display(perm.head(30))

## 13. SHAP attribution

SHAP is reported separately from held-out permutation importance and is used to describe the corresponding full-data reference-model refits. Native attribution scales are retained; no composite XAI score is formed.

In [ ]:
shap=read_csv('results/revision_comments_8_R1_17_18_R2/SHAP_Separate_Summary_Comments_17-R2_8-R1.csv')
ranks=read_csv('results/revision_comments_8_R1_17_18_R2/Rank_Agreement_Excluding_Pr_Comment_8-R1.csv')
display(shap.head(30))
display(ranks)

## 14. Candidate measured-domain screen

The final screen is based on measured source-study values, not predicted Pr. Conditions must be jointly observed in printability and angular-fidelity data, with measured Pr in 0.95-1.05 and at least one measured AF in 1.00-1.25. The final set is unordered.

In [ ]:
run_script('src/revision_comments_2_R1_6_7_R2_candidate_set.py')
audit=read_csv('results/revision_comments_2_R1_6_7_R2/Candidate_Screening_Audit_Comments_2-R1_6-7-R2.csv')
cands=read_csv('results/revision_comments_2_R1_6_7_R2/Table4_Unordered_Candidate_Set_Comments_2-R1_6-7-R2.csv')
display(audit)
display(cands)

## 15. Pr-threshold sensitivity

The measured-Pr tolerance is relaxed from 0.95-1.05 to 0.90-1.10 while the measured-AF criterion is held fixed. This sensitivity does not redefine the primary candidate set.

In [ ]:
run_script('src/revision_comment_7_R2_pr_gate_sensitivity.py')
prs=read_csv('results/revision_comment_7_R2_pr_gate_sensitivity/Pr_Gate_Sensitivity_Summary_Comment_7_R2.csv')
added=read_csv('results/revision_comment_7_R2_pr_gate_sensitivity/Table_S31_Relaxed_Pr_Only_Conditions_Comment_7_R2.csv')
display(prs)
display(added)

## 16. Bootstrap ranking sensitivity

This is the one explicitly historical analysis. It re-evaluates the 17 originally submitted strict candidates under 1,000 formulation-cluster bootstrap replicates. The archived candidate list is used only to define that retrospective set. The bootstrap uses the explicit fixed seed preserved by the revision analysis.

In [ ]:
historical = ROOT/'archive'/'legacy_pre_revision_candidate_ranking'/'tables'/'029_tables_final_top_strict_design_candidates.csv'
assert historical.exists(), 'Documented historical candidate artifact is missing.'
run_script('src/revision_comments_2_R1_6_R2_bootstrap_ranking.py')
bs=read_csv('results/revision_comments_2_R1_6_R2_bootstrap_ranking/Bootstrap_Candidate_Ranking_Summary.csv')
pair=read_csv('results/revision_comments_2_R1_6_R2_bootstrap_ranking/Bootstrap_Pairwise_Differences_vs_Original_Top.csv')
display(bs.head(20))
display(pair.head(20))

## 17. Table 2 / Table S19 reconciliation

Selection-aware nested performance and the submitted representation-sensitivity analysis are different analyses. This reconciliation records the nested estimate, modal nested selection, S19 model identities, S19 R² values, and whether the S19 comparison holds the algorithm constant.

In [ ]:
run_script('src/revision_comment_6_R1_reconciliation.py')
run_script('src/revision_comment_8_R2_reconciliation_audit.py')
rec=read_csv('results/revision_comment_6_R1_reconciliation/Table_S27_Table2_vs_TableS19_Reconciliation.csv')
display(rec)

## 18. Final tables, figures, and exports

The final asset step regenerates scientific figures and release-facing analysis artifacts from the current outputs. The notebook then performs the same mandatory numerical assertions used by the master runner for the core manuscript/SI quantities.

In [ ]:
run_script('src/generate_descriptive_figures.py')
run_script('src/finalize_revision_assets.py')

# Compact mandatory numerical verification from freshly generated outputs.
def approx(a,b,tol=5e-4):
    return abs(float(a)-float(b)) <= tol

# Nested performance
rp=rep.set_index('Target')
checks=[]
# infer likely columns by normalized names
def pick(df, target, tokens):
    row=df.set_index('Target').loc[target]
    for c in df.columns:
        n=c.lower().replace('²','2').replace('^','').replace(' ','_')
        if all(t in n for t in tokens): return row[c]
    raise KeyError((target,tokens,list(df.columns)))

for t,v in {'Pr':-0.083,'SR':-0.074,'Qm':0.762,'AF':0.712}.items():
    val=pick(rep,t,['repeated','r2'])
    checks.append((f'Nested repeated {t}',float(val),v,approx(val,v)))
for t,v in {'Pr':-0.348,'SR':0.393,'Qm':-1.458,'AF':0.762}.items():
    val=pick(rep,t,['lofo','r2'])
    checks.append((f'Nested LOFO {t}',float(val),v,approx(val,v)))

# Repeat-pooled R2
for t,v in {'Pr':0.321,'SR':0.033,'Qm':0.846,'AF':0.742}.items():
    row=r2agg[r2agg['Target']==t].iloc[0]
    val=row['Repeat_pooled_OOF_R2_mean']
    checks.append((f'Repeat-pooled {t}',float(val),v,approx(val,v)))

# Table S27 reconciliation convention guard
s27_check=read_csv('results/revision_comment_6_R1_reconciliation/Table_S27_Table2_vs_TableS19_Reconciliation.csv')
for t in ['Pr','SR','Qm','AF']:
    rr=s27_check[(s27_check['Target']==t)&(s27_check['Validation']=='Repeated_KFold')]['Table2_nested_R2'].iloc[0]
    rl=s27_check[(s27_check['Target']==t)&(s27_check['Validation']=='LOFO')]['Table2_nested_R2'].iloc[0]
    expected_rr=rp.loc[t,'Nested repeated R2 mean']
    expected_rl=rp.loc[t,'Nested LOFO R2']
    checks.append((f'Table S27 repeated convention {t}',float(rr),float(expected_rr),abs(float(rr)-float(expected_rr))<=1e-12))
    checks.append((f'Table S27 LOFO convention {t}',float(rl),float(expected_rl),abs(float(rl)-float(expected_rl))<=1e-12))

# Reviewer-analysis reconciliation export must use the same Table 2 convention.
s19rec=read_csv('results/revision_comments_6_7_R1_8_R2/Table2_vs_TableS19_Reconciliation_Comments_6-7-R1_8-R2.csv')
for t in ['Pr','SR','Qm','AF']:
    rr=s19rec[(s19rec['Target']==t)&(s19rec['Validation']=='Repeated_KFold')]['Table2_nested_R2'].iloc[0]
    rl=s19rec[(s19rec['Target']==t)&(s19rec['Validation']=='LOFO')]['Table2_nested_R2'].iloc[0]
    expected_rr=rp.loc[t,'Nested repeated R2 mean']
    expected_rl=rp.loc[t,'Nested LOFO R2']
    checks.append((f'S19 reconciliation repeated convention {t}',float(rr),float(expected_rr),abs(float(rr)-float(expected_rr))<=1e-12))
    checks.append((f'S19 reconciliation LOFO convention {t}',float(rl),float(expected_rl),abs(float(rl)-float(expected_rl))<=1e-12))



# Table S30 exact matched-partition and corrected-value checks
s30=read_csv('results/revision_comment_3_R2_baseline_contrast/Baseline_vs_Nested_Apples_to_Apples_Comment_3-R2.csv')
def s30val(target, baseline, col):
    row=s30[(s30['Target']==target)&(s30['Baseline']==baseline)&s30['Validation'].str.contains('repeat-pooled',case=False,na=False)].iloc[0]
    return float(row[col])
for name,actual,expected in [
    ('S30 AF categorical baseline repeat-pooled R2', s30val('AF','AF: angle-only categorical mean','Baseline_R2_mean'), 0.793),
    ('S30 nested AF repeat-pooled R2', s30val('AF','AF: angle-only categorical mean','Nested_multivariable_R2_mean'), 0.742),
    ('S30 Qm power-law baseline repeat-pooled R2', s30val('Qm','Qm: P/TP power law','Baseline_R2_mean'), -0.027),
    ('S30 nested Qm repeat-pooled R2', s30val('Qm','Qm: P/TP power law','Nested_multivariable_R2_mean'), 0.846),
    ('S30 AF nested-lower-RMSE fraction', s30val('AF','AF: angle-only categorical mean','Fraction_repeats_nested_better_RMSE'), 0.0),
    ('S30 Qm nested-lower-RMSE fraction', s30val('Qm','Qm: P/TP power law','Fraction_repeats_nested_better_RMSE'), 1.0),
]:
    checks.append((name,actual,expected,approx(actual,expected,1e-12 if 'fraction' in name else 5e-4)))

bp=read_csv('results/revision_comment_3_R2_baseline_contrast/Baseline_Repeated_Predictions_Comment_3-R2.csv')
npred=read_csv('results/revision_comments_4_R1_16_R2/Repeated_Predictions_Comments_4-R1_16-R2.csv')
for t in ['AF','Qm']:
    bkeys=set(map(tuple,bp[bp['Target']==t][['Repeat','Outer_fold','Row_index']].astype(int).to_numpy()))
    nkeys=set(map(tuple,npred[npred['Target']==t][['Repeat','Outer_fold','Row_index']].astype(int).to_numpy()))
    checks.append((f'S30 exact repeated partitions {t}', int(bkeys==nkeys), 1, bkeys==nkeys))

check_df=pd.DataFrame(checks,columns=['Check','Generated','Expected','PASS'])
display(check_df)
assert check_df['PASS'].all(), 'One or more mandatory notebook checks failed.'
print('Notebook scientific path completed with mandatory core checks passing.')

### Notebook completion criterion

A successful **Run All** means every cell above executed sequentially in a fresh kernel, every canonical analysis script completed without error, and the final mandatory assertions passed. This notebook is not by itself the release certification: the final release still requires the separate clean-room master-runner execution, notebook Run All in that clean copy, manuscript/SI cross-check, reproducibility report, and final package audit.